# 02 — Review Analytics: 10 SQL-Style Queries on 67K Reviews

**Data Source:** UCSD Amazon Review Dataset — Electronics 5-core subset
- 67,325 real reviews | 27,832 unique products | 53,609 unique reviewers
- Date range: 1999-11-23 → 2014-07-23


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

Path("figures").mkdir(exist_ok=True)

# Load data
reviews = pd.read_csv("data/amazon_reviews_electronics_5core.csv")
reviews["reviewDate"] = pd.to_datetime(reviews["unixReviewTime"], unit="s")
reviews["reviewYear"] = reviews["reviewDate"].dt.year
reviews["reviewMonth"] = reviews["reviewDate"].dt.month
reviews["reviewLength"] = reviews["reviewText"].fillna("").str.len()
reviews["helpfulnessRatio"] = np.where(
    reviews["helpful_total"] > 0,
    reviews["helpful_upvotes"] / reviews["helpful_total"],
    np.nan,
)

print("=== DATA LOADED ===")
print(f"Reviews: {len(reviews):,} | Products: {reviews['asin'].nunique():,} | Reviewers: {reviews['reviewerID'].nunique():,}")

# Q1: Top Products by Review Volume
print("\n=== Q1: Top Products by Volume ===")
q1 = (
    reviews.groupby("asin")
    .agg(review_volume=("overall", "size"), avg_rating=("overall", "mean"))
    .reset_index()
    .sort_values("review_volume", ascending=False)
    .head(10)
)
print(q1.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(q1)), q1["review_volume"], color="steelblue")
ax.set_yticks(range(len(q1)))
ax.set_yticklabels([f"{a[:12]}..." for a in q1["asin"]])
ax.invert_yaxis()
ax.set_xlabel("Review Volume", fontsize=12)
ax.set_title("Q1 — Top 10 Products by Review Volume", fontsize=14, fontweight="bold")
for i, vol in enumerate(q1["review_volume"]):
    ax.text(vol + 1, i, str(int(vol)), va="center", fontsize=10)
plt.tight_layout()
fig.savefig("figures/figure_008_q1_top_volume.png", dpi=150, bbox_inches="tight")
plt.show()

# Q2: Rating Distribution by Year
print("\n=== Q2: Rating Distribution by Year ===")
yearly_rating = (
    reviews.groupby(["reviewYear", "overall"])
    .size()
    .reset_index(name="count")
    .pivot(index="reviewYear", columns="overall", values="count")
    .fillna(0)
)
# Normalize to percentages
yearly_pct = yearly_rating.div(yearly_rating.sum(axis=1), axis=0) * 100
print(yearly_pct.round(1).to_string())

fig, ax = plt.subplots(figsize=(12, 7))
yearly_pct.plot(kind="bar", stacked=True, ax=ax, color=["#d32f2f", "#f57c00", "#fbc02d", "#689f38", "#388e3c"], width=0.8)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Percentage of Reviews", fontsize=12)
ax.set_title("Q2 — Rating Distribution by Year (Stacked %)", fontsize=14, fontweight="bold")
ax.legend(title="Rating", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, 100)
plt.tight_layout()
fig.savefig("figures/figure_009_q2_rating_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

# Q3: Helpfulness Leaderboard
print("\n=== Q3: Most Helpful Reviewers ===")
q3 = (
    reviews[reviews["helpful_total"] > 0]
    .groupby("reviewerID")
    .agg(total_votes=("helpful_total", "sum"), upvotes=("helpful_upvotes", "sum"), review_count=("asin", "size"))
    .reset_index()
)
q3["helpfulness_rate"] = q3["upvotes"] / q3["total_votes"]
q3 = q3[q3["total_votes"] >= 10].sort_values("helpfulness_rate", ascending=False).head(10)
print(q3.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(q3)), q3["helpfulness_rate"], color="seagreen")
ax.set_yticks(range(len(q3)))
ax.set_yticklabels([r[:12] + "..." for r in q3["reviewerID"]])
ax.invert_yaxis()
ax.set_xlabel("Helpfulness Rate", fontsize=12)
ax.set_title("Q3 — Most Helpful Reviewers (≥10 votes)", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y*100)}%"))
plt.tight_layout()
fig.savefig("figures/figure_010_q3_helpful_reviewers.png", dpi=150, bbox_inches="tight")
plt.show()

# Q4: Review Length vs Rating Correlation
print("\n=== Q4: Review Length vs. Rating ===")
length_rating = reviews.groupby("overall")["reviewLength"].agg(["mean", "median", "std"]).reset_index()
print(length_rating.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(length_rating["overall"], length_rating["mean"], color=["#d32f2f", "#f57c00", "#fbc02d", "#689f38", "#388e3c"], alpha=0.7, label="Mean")
ax.plot(length_rating["overall"], length_rating["median"], marker="o", color="darkblue", linewidth=2, markersize=8, label="Median")
ax.set_xlabel("Star Rating", fontsize=12)
ax.set_ylabel("Review Length (characters)", fontsize=12)
ax.set_title("Q4 — Review Length vs. Rating (Mean vs. Median)", fontsize=14, fontweight="bold")
ax.legend()
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
fig.savefig("figures/figure_011_q4_length_rating.png", dpi=150, bbox_inches="tight")
plt.show()

# Q5: Seasonal Patterns (already done in nb01, do a different angle)
print("\n=== Q5: Month-of-Year Aggregation ===")
q5 = reviews.groupby("reviewMonth").size().reset_index(name="volume")
print(q5.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
colors = ["lightsteelblue"] * 12
colors[11] = "#d32f2f"  # December highlight
colors[0] = "#f57c00"   # January highlight
ax.bar(range(1, 13), q5["volume"], color=colors, edgecolor="white", linewidth=1.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_ylabel("Total Reviews", fontsize=12)
ax.set_xlabel("Month", fontsize=12)
ax.set_title("Q5 — Reviews by Month (Dec & Jan Peaks)", fontsize=14, fontweight="bold")
for i, v in enumerate(q5["volume"]):
    ax.text(i+1, v + 100, f"{int(v):,}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_012_q5_monthly_aggregate.png", dpi=150, bbox_inches="tight")
plt.show()

# Q6: Reviewer Loyalty Distribution
print("\n=== Q6: Reviewer Loyalty ===")
q6 = reviews["reviewerID"].value_counts().reset_index()
q6.columns = ["reviewerID", "review_count"]
loyalty = q6["review_count"].value_counts().sort_index().reset_index()
loyalty.columns = ["reviews_per_reviewer", "reviewer_count"]
print(loyalty.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(loyalty["reviews_per_reviewer"].head(15), loyalty["reviewer_count"].head(15), color="coral", edgecolor="white")
ax.set_xlabel("Reviews Written per Reviewer", fontsize=12)
ax.set_ylabel("Number of Reviewers", fontsize=12)
ax.set_title("Q6 — Reviewer Loyalty (5-core: minimum 5 reviews each)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_013_q6_reviewer_loyalty.png", dpi=150, bbox_inches="tight")
plt.show()

# Q7: Summary Usage by Rating
print("\n=== Q7: Summary Usage by Rating ===")
reviews["hasSummary"] = reviews["summary"].notna() & (reviews["summary"].str.len() > 0)
q7 = reviews.groupby("overall")["hasSummary"].mean().reset_index()
q7.columns = ["rating", "summary_rate"]
print(q7.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(q7["rating"], q7["summary_rate"], color=["#d32f2f", "#f57c00", "#fbc02d", "#689f38", "#388e3c"], edgecolor="white")
ax.set_xlabel("Star Rating", fontsize=12)
ax.set_ylabel("% with Summary", fontsize=12)
ax.set_title("Q7 — Summary Usage by Rating", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y*100)}%"))
for i, r in enumerate(q7["summary_rate"]):
    ax.text(q7.iloc[i]["rating"], r + 0.02, f"{r:.0%}", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_014_q7_summary_usage.png", dpi=150, bbox_inches="tight")
plt.show()

# Q8: Helpfulness by Length (already done in nb01, show table)
print("\n=== Q8: Helpfulness by Length Tier ===")
reviews["length_tier"] = pd.cut(
    reviews["reviewLength"],
    bins=[0, 100, 500, float("inf")],
    labels=["Short (<100)", "Medium (100-500)", "Long (500+)"],
)
q8 = (
    reviews[reviews["helpful_total"] > 0]
    .groupby("length_tier")
    .agg(avg_helpfulness=("helpfulnessRatio", "mean"), count=("asin", "size"))
    .reset_index()
)
print(q8.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(q8["length_tier"], q8["avg_helpfulness"], color=["#fbc02d", "#689f38", "#388e3c"], edgecolor="white", linewidth=1.5)
ax.set_ylabel("Helpfulness Ratio", fontsize=12)
ax.set_xlabel("Review Length Tier", fontsize=12)
ax.set_title("Q8 — Helpfulness by Review Length", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y*100)}%"))
for bar, h, c in zip(ax.patches, q8["avg_helpfulness"], q8["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{h:.0%}\n({c:,})", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_015_q8_helpfulness_length.png", dpi=150, bbox_inches="tight")
plt.show()

# Q9: Year-over-Year Growth
print("\n=== Q9: Year-over-Year Growth ===")
q9 = reviews.groupby("reviewYear").size().reset_index(name="volume")
q9["yoy_growth"] = q9["volume"].pct_change() * 100
print(q9.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["green" if g > 0 else "red" for g in q9["yoy_growth"].fillna(0)]
ax.bar(q9["reviewYear"], q9["yoy_growth"].fillna(0), color=colors, alpha=0.7, edgecolor="white")
ax.axhline(y=0, color="black", linewidth=1)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("YoY Growth (%)", fontsize=12)
ax.set_title("Q9 — Year-over-Year Review Volume Growth", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_016_q9_yoy_growth.png", dpi=150, bbox_inches="tight")
plt.show()

# Q10: Product Lifecycle — First review to peak
print("\n=== Q10: Product Lifecycle (Sample) ===")
product_dates = (
    reviews.groupby("asin")
    .agg(first_review=("reviewDate", "min"), last_review=("reviewDate", "max"), volume=("asin", "size"))
    .reset_index()
    .sort_values("volume", ascending=False)
    .head(10)
)
product_dates["lifespan_days"] = (product_dates["last_review"] - product_dates["first_review"]).dt.days
print(product_dates.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(product_dates["volume"], product_dates["lifespan_days"], alpha=0.6, color="purple", s=100)
ax.set_xlabel("Total Reviews", fontsize=12)
ax.set_ylabel("Lifespan (days)", fontsize=12)
ax.set_title("Q10 — Product Lifespan vs. Review Volume (Top 10)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/figure_017_q10_lifecycle.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== ALL 10 QUERIES COMPLETE ===")
